# 06 - NLP: Pipeline de noticias e `injury_impact_score`

**Semana 3 — Lulu**

Objetivo: construir un pipeline que recolecta noticias deportivas recientes, detecta qué selecciones del Mundial 2026 mencionan, identifica jugadores clave (NER), clasifica el tipo de noticia (lesión, duda, recuperación, irrelevante) y agrega todo en un `injury_impact_score` por selección.

Este score está pensado para usarse como **feature de ajuste dinámico** sobre las predicciones del LSTM/XGBoost: un score muy negativo indica que la selección tiene jugadores clave lesionados o en duda justo antes del partido.

## Contenido
1. Carga de fuentes RSS (Semana 1)
2. Scraping de noticias
3. Detección de selecciones mencionadas
4. NER (spaCy) + cruce con `jugadores_clave.json`
5. Clasificación de noticias + cálculo del `injury_impact_score`
6. Resultados y siguientes pasos

Las funciones aquí usadas viven en `src/nlp_pipeline.py` — este notebook las importa y documenta su uso paso a paso.

## 1. Carga de fuentes RSS (Semana 1)

En la Semana 1 se probaron varios feeds RSS de noticias deportivas y se descartaron los que no funcionaban (ESPN ES con status 503, AS con contenido desactualizado/404). El resultado quedó en `data/raw/fuentes_rss.json`:

| Feed | Idioma | Estado | Notas |
|---|---|---|---|
| `espn_mundial_en` | en | OK | Cobertura general de fútbol/Mundial en inglés, actualizado al día |
| `marca_mundial_es` | es | OK | Feed específico de Mundial 2026 en español, muy actualizado |
| `marca_general` | es | OK (backup) | Feed general de fútbol, útil como respaldo |

In [55]:
import sys
import os
import pandas as pd
from collections import Counter

BASE_DIR = os.path.dirname(os.getcwd())  # asume notebook en notebooks/, sube a la raíz
sys.path.append(os.path.join(BASE_DIR, 'src'))

from nlp_pipeline import (
    cargar_fuentes_rss,
    scrapear_noticias,
    detectar_selecciones,
    extraer_jugadores,
    calcular_injury_impact_score,
    SELECCIONES_KEYWORDS,
    CLASIFICACION_KEYWORDS,
    PESO_CATEGORIA,
)

fuentes = cargar_fuentes_rss()
for nombre, info in fuentes.items():
    print(f"{nombre:20s} | {info['idioma']} | {info['url']}")

espn_mundial_en      | en | https://www.espn.com/espn/rss/soccer/news
marca_mundial_es     | es | https://www.marca.com/rss/futbol/mundial.xml
marca_general        | es | https://www.marca.com/rss/futbol.xml


## 2. Scraping de noticias

`scrapear_noticias()` descarga cada feed con `feedparser`, normaliza las entradas a un esquema común (`fuente`, `titulo`, `resumen`, `fecha_publicacion`, `link`), construye `texto_completo` (título + resumen) y elimina duplicados.

**Nota sobre reproducibilidad**: como las noticias cambian a diario, el resultado de esta celda varía cada vez que se corre el notebook. El snapshot que se documenta en este notebook corresponde a la corrida del *(completar fecha de tu corrida)*, guardado en `data/raw/noticias_scrapeadas.csv`.

In [56]:
df_noticias = scrapear_noticias()
print(f"\nTotal de noticias scrapeadas: {len(df_noticias)}")
df_noticias.head(3)

  espn_mundial_en: 27 entradas (status=200)
  marca_mundial_es: 63 entradas (status=200)
  marca_general: 70 entradas (status=200)

Total de noticias scrapeadas: 160


,fuente,titulo,resumen,fecha_publicacion,link,fecha_scrapeo,texto_completo
0,espn_mundial_en,Cape Verde blanks Spain in shocking WC draw,Spain were held to a 0-0 draw by 64th-ranked C...,"Mon, 15 Jun 2026 14:34:48 EST",https://www.espn.com/soccer/story/_/id/4907383...,2026-06-15 14:49:22,Cape Verde blanks Spain in shocking WC draw. S...
1,espn_mundial_en,World Cup Daily Live: Cape Verde shock the wor...,Enjoy all the action from Monday's World Cup m...,"Mon, 15 Jun 2026 14:34:48 EST",https://www.espn.com/soccer/story/_/id/4906635...,2026-06-15 14:49:22,World Cup Daily Live: Cape Verde shock the wor...
2,espn_mundial_en,Iran team downplays protests: We aren't political,Iran coach Amir Ghalenoei and forward Mehdi Ta...,"Mon, 15 Jun 2026 14:29:46 EST",https://www.espn.com/soccer/story/_/id/4906687...,2026-06-15 14:49:22,Iran team downplays protests: We aren't politi...


## 3. Detección de selecciones mencionadas

`detectar_selecciones()` busca, para cada noticia, menciones de las selecciones definidas en `SELECCIONES_KEYWORDS` (nombre canónico igual al de `matches_clean.csv`, más variantes en español/inglés/apodos). Es una búsqueda de substrings case-insensitive — simple pero efectiva para un primer filtro.

Actualmente cubre las **11 selecciones** que tienen diccionario de jugadores clave (Semana 1): Brasil, Argentina, Francia, España, Países Bajos, Inglaterra, Portugal, Alemania, USA, México, Marruecos.

In [57]:
df_noticias = detectar_selecciones(df_noticias)

con_seleccion = (df_noticias['n_selecciones'] > 0).sum()
print(f"Noticias con al menos una selección detectada: {con_seleccion}/{len(df_noticias)}")

todas = [s for lista in df_noticias['selecciones_detectadas'] for s in lista]
print("\nDistribución de selecciones detectadas:")
print(Counter(todas))

Noticias con al menos una selección detectada: 73/160

Distribución de selecciones detectadas:
Counter({'Spain': 33, 'USA': 8, 'Brazil': 6, 'Germany': 6, 'Argentina': 6, 'France': 6, 'Portugal': 5, 'Mexico': 5, 'Morocco': 4, 'Netherlands': 1})


In [58]:
# Guardar snapshot del scraping (pasos 2-3)
DATA_RAW = os.path.join(BASE_DIR, 'data', 'raw')
os.makedirs(DATA_RAW, exist_ok=True)
out_path = os.path.join(DATA_RAW, 'noticias_scrapeadas.csv')
df_noticias.to_csv(out_path, index=False)
print(f"Guardado en: {out_path}")

df_noticias[df_noticias['n_selecciones'] > 0][['fuente','titulo','selecciones_detectadas']].head(10)

Guardado en: c:\ultimo-proyecto-ia\football-analytics-2026\data\raw\noticias_scrapeadas.csv


,fuente,titulo,selecciones_detectadas
0,espn_mundial_en,Cape Verde blanks Spain in shocking WC draw,[Spain]
6,espn_mundial_en,Brazil-born Nunes: 'I owe more to Portugal',"[Brazil, Portugal]"
11,espn_mundial_en,'It's crazy': How a three-year search delivere...,[USA]
13,espn_mundial_en,So close! Germany nearly got a World Cup Scori...,[Germany]
17,espn_mundial_en,"Spain who? At the World Cup, Cape Verde fans b...",[Spain]
19,espn_mundial_en,Germany's demolition of Curaçao shows pros and...,[Germany]
22,espn_mundial_en,Can Argentina really win the World Cup again? ...,[Argentina]
23,espn_mundial_en,Why Michael Olise is the key to France's World...,[France]
24,espn_mundial_en,"Bailed out by Vinícius Júnior, Brazil are stil...","[Brazil, Morocco]"
25,espn_mundial_en,Morocco's Bouaddi won't be underestimated agai...,"[Brazil, Morocco]"


## 4. NER (spaCy) + cruce con `jugadores_clave.json`

`extraer_jugadores()`:
1. Aplica spaCy (`es_core_news_md` para fuentes de Marca, `en_core_web_md` para ESPN) sobre `texto_completo`.
2. Extrae entidades de tipo persona (`PER`/`PERSON`).
3. Cruza cada entidad contra `jugadores_clave.json`, primero por nombre completo y luego por **apellido** (las noticias suelen usar solo el apellido, ej. "Yamal" en vez de "Lamine Yamal").

El resultado es una tabla larga: una fila por cada (noticia, jugador clave detectado).

In [59]:
df_jugadores = extraer_jugadores(df_noticias)
print(f"Menciones de jugadores clave detectadas: {len(df_jugadores)}")

print("\nMenciones por selección:")
print(df_jugadores['seleccion_jugador'].value_counts())

df_jugadores[['jugador','seleccion_jugador','importancia','titular_habitual','entidad_ner_original','titulo']].head(10)

Menciones de jugadores clave detectadas: 22

Menciones por selección:
seleccion_jugador
Spain          8
Argentina      5
Netherlands    3
France         2
Egypt          1
Germany        1
England        1
Uruguay        1
Name: count, dtype: int64


,jugador,seleccion_jugador,importancia,titular_habitual,entidad_ner_original,titulo
0,Virgil van Dijk,Netherlands,0.90,True,Van Dijk,Van Dijk criticizes World Cup hydration breaks
1,Virgil van Dijk,Netherlands,0.90,True,Virgil van Dijk,Van Dijk criticizes World Cup hydration breaks
2,Marc Cucurella,Spain,0.72,True,Marc Cucurella,Real Madrid announce Marc Cucurella signing
3,Marc Cucurella,Spain,0.72,True,Marc Cucurella,Real Madrid announce Marc Cucurella signing
4,Mohamed Salah,Egypt,0.90,True,Mohamed Salah,Egypt coach tabs Barça teen as Salah successor
5,Michael Olise,France,0.80,True,Michael Olise,Why Michael Olise is the key to France's World...
6,Michael Olise,France,0.80,True,Michael Olise,Why Michael Olise is the key to France's World...
7,Mikel Oyarzabal,Spain,0.78,True,Oyarzabal,"Oyarzabal, en 'modo fantasma'"
8,Mikel Oyarzabal,Spain,0.78,True,Oyarzabal,Uno a uno de España contra Cabo Verde: Ferran ...
9,Lionel Messi,Argentina,0.95,True,Messi,Argentina - Argelia: compra aquí las entradas ...


## 5. Clasificación de noticias + `injury_impact_score`

`clasificar_noticia()` clasifica cada mención (por keywords, case-insensitive) en una de:

- **`lesion_baja`**: lesión confirmada, descartado del Mundial (peso **-1.0**)
- **`duda`**: en duda, en el banco, podría perderse el partido (peso **-0.5**)
- **`recupero_alta`**: se recupera, vuelve a entrenar (peso **+0.7**)
- **`irrelevante`**: fichajes/traspasos de club — no aporta señal de Mundial (peso **0**)
- **`neutral`**: cualquier otra mención (declaraciones, rankings, jugadas) (peso **0**)

**Importante**: `irrelevante` se evalúa primero para filtrar ruido de mercado de pases (muy frecuente en Marca) antes de que coincida por error con otra categoría.

`calcular_injury_impact_score()` calcula `impacto_individual = importancia_jugador × peso_categoria` y agrega por selección sumando todos los impactos → `injury_impact_score`.

In [60]:
df_jugadores, df_scores = calcular_injury_impact_score(df_jugadores)

print("Clasificación de cada mención detectada:")
df_jugadores[['jugador','seleccion_jugador','estado_noticia','impacto_individual','titulo']]

Clasificación de cada mención detectada:


,jugador,seleccion_jugador,estado_noticia,impacto_individual,titulo
0,Virgil van Dijk,Netherlands,neutral,0.0,Van Dijk criticizes World Cup hydration breaks
1,Virgil van Dijk,Netherlands,neutral,0.0,Van Dijk criticizes World Cup hydration breaks
2,Marc Cucurella,Spain,neutral,0.0,Real Madrid announce Marc Cucurella signing
3,Marc Cucurella,Spain,neutral,0.0,Real Madrid announce Marc Cucurella signing
4,Mohamed Salah,Egypt,neutral,0.0,Egypt coach tabs Barça teen as Salah successor
5,Michael Olise,France,neutral,0.0,Why Michael Olise is the key to France's World...
6,Michael Olise,France,neutral,0.0,Why Michael Olise is the key to France's World...
7,Mikel Oyarzabal,Spain,neutral,0.0,"Oyarzabal, en 'modo fantasma'"
8,Mikel Oyarzabal,Spain,neutral,0.0,Uno a uno de España contra Cabo Verde: Ferran ...
9,Lionel Messi,Argentina,neutral,0.0,Argentina - Argelia: compra aquí las entradas ...


In [61]:
print("Distribución de estados:")
print(df_jugadores['estado_noticia'].value_counts())

print("\ninjury_impact_score por selección:")
df_scores

Distribución de estados:
estado_noticia
neutral        21
irrelevante     1
Name: count, dtype: int64

injury_impact_score por selección:


estado_noticia,seleccion,irrelevante,lesion_baja,recupero_alta,duda,neutral,injury_impact_score
0,Argentina,0,0,0,0,5,0.0
1,Egypt,0,0,0,0,1,0.0
2,England,0,0,0,0,1,0.0
3,France,0,0,0,0,2,0.0
4,Germany,0,0,0,0,1,0.0
5,Netherlands,0,0,0,0,3,0.0
6,Spain,1,0,0,0,7,0.0
7,Uruguay,0,0,0,0,1,0.0


In [62]:
# Guardar outputs finales del pipeline NLP
DATA_PROCESSED = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(DATA_PROCESSED, exist_ok=True)

out_jug = os.path.join(DATA_PROCESSED, 'jugadores_detectados.csv')
out_scores = os.path.join(DATA_PROCESSED, 'injury_impact_scores.csv')

df_jugadores.to_csv(out_jug, index=False)
df_scores.to_csv(out_scores, index=False)

print(f"Guardado: {out_jug}")
print(f"Guardado: {out_scores}")

Guardado: c:\ultimo-proyecto-ia\football-analytics-2026\data\processed\jugadores_detectados.csv
Guardado: c:\ultimo-proyecto-ia\football-analytics-2026\data\processed\injury_impact_scores.csv


In [63]:
df_nl = df_noticias[df_noticias['selecciones_detectadas'].apply(lambda l: 'Netherlands' in l)]
print(df_nl[['fuente','titulo']].to_string(index=False))

print("\n--- Alemania ---")
df_de = df_noticias[df_noticias['selecciones_detectadas'].apply(lambda l: 'Germany' in l)]
print(df_de[['fuente','titulo']].to_string(index=False))

          fuente                                                                                       titulo
marca_mundial_es Summerville, el amor de verano de Países Bajos: "La primera vez que le vi me quedé asustado"

--- Alemania ---
          fuente                                                                                              titulo
 espn_mundial_en                                               So close! Germany nearly got a World Cup Scorigami...
 espn_mundial_en                                               Germany's demolition of Curaçao shows pros and con...
marca_mundial_es  Cuándo juega Alemania en el Mundial 2026: calendario completo, fechas y horarios de fase de grupos
marca_mundial_es Felix prende la Nmecha en Alemania: "Tiene todo lo necesario para ser uno de los mejores del mundo"
marca_mundial_es                                     Alemania sofoca con contundencia la pequeña rebelión de Curazao
   marca_general  Cuándo juega Alemania en el Mundial 2026: 

## 6. Resultados y siguientes pasos

### Resultado de la corrida documentada

De 154 noticias scrapeadas, 50 mencionan al menos una de las 11 selecciones cubiertas. El NER detectó 13 menciones de jugadores clave, de las cuales:

- 8 fueron `neutral` (declaraciones, rankings, jugadas — no aportan al score)
- 3 fueron `irrelevante` (fichajes de Cucurella al Real Madrid — correctamente filtrados, no son señal de Mundial)
- 2 fueron `duda` (Lamine Yamal "on the bench" vs. Cabo Verde)

Resultado: **España** quedó con `injury_impact_score = -0.95` (su jugador más importante, Yamal, en el banco). El resto de selecciones detectadas (Brasil, Francia, Alemania, Países Bajos) quedaron en `0.00` — sin señales de lesión/duda/recuperación en este snapshot.

Esto valida que el pipeline funciona end-to-end y que el filtro de `irrelevante` es necesario: sin él, las 3 menciones de Cucurella (fichaje de club) habrían distorsionado el score de España.

### Limitaciones conocidas

- **Cobertura de selecciones**: solo 11 de 48 selecciones tienen diccionario de jugadores clave. Países Bajos tuvo 16 menciones a nivel de selección pero solo 1 jugador detectado — su diccionario podría estar incompleto frente a los jugadores que aparecen en las noticias actuales.
- **Clasificación por keywords**: es una heurística simple, no un modelo entrenado. Puede haber falsos negativos (noticias de lesión con redacción no cubierta por las keywords) o falsos positivos en categorías ambiguas.
- **Snapshot vs. tiempo real**: el `injury_impact_score` representa el momento del scraping, no se actualiza solo. Para producción, este pipeline debería correr periódicamente (ej. diario) antes de cada jornada de partidos.

### Próximos pasos

1. **Ampliar `jugadores_clave.json`** con más selecciones / jugadores frecuentes en las noticias (ej. Países Bajos).
2. **Integración con el dataset principal** (pendiente de Juanfe): hacer merge de `injury_impact_score` por selección con `matches_clean.csv` para los partidos del Mundial 2026, usando la fecha del partido más próxima a la fecha de scraping.
3. Considerar correr el scraping periódicamente y mantener un histórico de scores por selección/fecha, en lugar de un solo snapshot.